In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score,f1_score,recall_score,precision_score,confusion_matrix,classification_report
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv(r'C:\Users\fatima\Desktop\New folder\WA_Fn-UseC_-Telco-Customer-Churn.csv')
df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [3]:
df.drop_duplicates(inplace=True)
df.drop(columns='customerID',inplace=True)

In [4]:
df.shape

(7043, 20)

In [5]:
df.replace({'Yes':1,'No':0},inplace=True)
df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,1,0,1,0,No phone service,DSL,0,1,0,0,0,0,Month-to-month,1,Electronic check,29.85,29.85,0
1,Male,0,0,0,34,1,0,DSL,1,0,1,0,0,0,One year,0,Mailed check,56.95,1889.5,0
2,Male,0,0,0,2,1,0,DSL,1,1,0,0,0,0,Month-to-month,1,Mailed check,53.85,108.15,1
3,Male,0,0,0,45,0,No phone service,DSL,1,0,1,1,0,0,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,0,0,2,1,0,Fiber optic,0,0,0,0,0,0,Month-to-month,1,Electronic check,70.70,151.65,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,1,1,24,1,1,DSL,1,0,1,1,1,1,One year,1,Mailed check,84.80,1990.5,0
7039,Female,0,1,1,72,1,1,Fiber optic,0,1,1,0,1,1,One year,1,Credit card (automatic),103.20,7362.9,0
7040,Female,0,1,1,11,0,No phone service,DSL,1,0,0,0,0,0,Month-to-month,1,Electronic check,29.60,346.45,0
7041,Male,1,1,0,4,1,1,Fiber optic,0,0,0,0,0,0,Month-to-month,1,Mailed check,74.40,306.6,1


In [6]:
df=pd.get_dummies(df,columns=['MultipleLines','InternetService','Contract','PaymentMethod','OnlineSecurity','OnlineBackup',
                              'DeviceProtection','TechSupport','StreamingTV','StreamingMovies'],dtype=int)
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,DeviceProtection_No internet service,TechSupport_0,TechSupport_1,TechSupport_No internet service,StreamingTV_0,StreamingTV_1,StreamingTV_No internet service,StreamingMovies_0,StreamingMovies_1,StreamingMovies_No internet service
0,Female,0,1,0,1,0,1,29.85,29.85,0,...,0,1,0,0,1,0,0,1,0,0
1,Male,0,0,0,34,1,0,56.95,1889.5,0,...,0,1,0,0,1,0,0,1,0,0
2,Male,0,0,0,2,1,1,53.85,108.15,1,...,0,1,0,0,1,0,0,1,0,0
3,Male,0,0,0,45,0,0,42.30,1840.75,0,...,0,0,1,0,1,0,0,1,0,0
4,Female,0,0,0,2,1,1,70.70,151.65,1,...,0,1,0,0,1,0,0,1,0,0


In [7]:
df.gender=df.gender.map({'Female':0,'Male':1})

In [8]:
pd.set_option('display.max_columns',None)

In [9]:
df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,MultipleLines_0,MultipleLines_1,MultipleLines_No phone service,InternetService_0,InternetService_DSL,InternetService_Fiber optic,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,OnlineSecurity_0,OnlineSecurity_1,OnlineSecurity_No internet service,OnlineBackup_0,OnlineBackup_1,OnlineBackup_No internet service,DeviceProtection_0,DeviceProtection_1,DeviceProtection_No internet service,TechSupport_0,TechSupport_1,TechSupport_No internet service,StreamingTV_0,StreamingTV_1,StreamingTV_No internet service,StreamingMovies_0,StreamingMovies_1,StreamingMovies_No internet service
0,0,0,1,0,1,0,1,29.85,29.85,0,0,0,1,0,1,0,1,0,0,0,0,1,0,1,0,0,0,1,0,1,0,0,1,0,0,1,0,0,1,0,0
1,1,0,0,0,34,1,0,56.95,1889.5,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,1,0,1,0,0,0,1,0,1,0,0,1,0,0,1,0,0
2,1,0,0,0,2,1,1,53.85,108.15,1,1,0,0,0,1,0,1,0,0,0,0,0,1,0,1,0,0,1,0,1,0,0,1,0,0,1,0,0,1,0,0
3,1,0,0,0,45,0,0,42.30,1840.75,0,0,0,1,0,1,0,0,1,0,1,0,0,0,0,1,0,1,0,0,0,1,0,0,1,0,1,0,0,1,0,0
4,0,0,0,0,2,1,1,70.70,151.65,1,1,0,0,0,0,1,1,0,0,0,0,1,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,1,0,1,1,24,1,1,84.80,1990.5,0,0,1,0,0,1,0,0,1,0,0,0,0,1,0,1,0,1,0,0,0,1,0,0,1,0,0,1,0,0,1,0
7039,0,0,1,1,72,1,1,103.20,7362.9,0,0,1,0,0,0,1,0,1,0,0,1,0,0,1,0,0,0,1,0,0,1,0,1,0,0,0,1,0,0,1,0
7040,0,0,1,1,11,0,1,29.60,346.45,0,0,0,1,0,1,0,1,0,0,0,0,1,0,0,1,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0
7041,1,1,1,0,4,1,1,74.40,306.6,1,0,1,0,0,0,1,1,0,0,0,0,0,1,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0


In [10]:
x=df.drop('Churn',axis=1).values
y=df['Churn'].values
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,shuffle=True)

In [11]:
df['TotalCharges']=df['TotalCharges'].replace(r'^\s*$',np.nan,regex=True)
df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='raise')
x=df.drop('Churn',axis=1).values
y=df['Churn'].values
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,shuffle=True)
a=x_train[:,8].astype('float')
df['TotalCharges']=df['TotalCharges'].fillna(1405.65)
df.isna().sum()

gender                                     0
SeniorCitizen                              0
Partner                                    0
Dependents                                 0
tenure                                     0
PhoneService                               0
PaperlessBilling                           0
MonthlyCharges                             0
TotalCharges                               0
Churn                                      0
MultipleLines_0                            0
MultipleLines_1                            0
MultipleLines_No phone service             0
InternetService_0                          0
InternetService_DSL                        0
InternetService_Fiber optic                0
Contract_Month-to-month                    0
Contract_One year                          0
Contract_Two year                          0
PaymentMethod_Bank transfer (automatic)    0
PaymentMethod_Credit card (automatic)      0
PaymentMethod_Electronic check             0
PaymentMet

In [12]:
# df=df.apply(pd.to_numeric,errors='raise')
df.dtypes

gender                                       int64
SeniorCitizen                                int64
Partner                                      int64
Dependents                                   int64
tenure                                       int64
PhoneService                                 int64
PaperlessBilling                             int64
MonthlyCharges                             float64
TotalCharges                               float64
Churn                                        int64
MultipleLines_0                              int32
MultipleLines_1                              int32
MultipleLines_No phone service               int32
InternetService_0                            int32
InternetService_DSL                          int32
InternetService_Fiber optic                  int32
Contract_Month-to-month                      int32
Contract_One year                            int32
Contract_Two year                            int32
PaymentMethod_Bank transfer (au

In [13]:
df.select_dtypes(include='object').columns

Index([], dtype='object')

In [14]:
df.isna().sum()

gender                                     0
SeniorCitizen                              0
Partner                                    0
Dependents                                 0
tenure                                     0
PhoneService                               0
PaperlessBilling                           0
MonthlyCharges                             0
TotalCharges                               0
Churn                                      0
MultipleLines_0                            0
MultipleLines_1                            0
MultipleLines_No phone service             0
InternetService_0                          0
InternetService_DSL                        0
InternetService_Fiber optic                0
Contract_Month-to-month                    0
Contract_One year                          0
Contract_Two year                          0
PaymentMethod_Bank transfer (automatic)    0
PaymentMethod_Credit card (automatic)      0
PaymentMethod_Electronic check             0
PaymentMet

In [15]:
x=df.drop('Churn',axis=1).values
y=df['Churn'].values
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,shuffle=True)

In [16]:
selector=SelectKBest(score_func=chi2,k='all')
selector.fit(x_train,y_train)
scores=pd.Series(selector.scores_,index=df.columns.drop('Churn')).sort_values(ascending=False)
print(scores)

TotalCharges                               474436.947247
tenure                                      12349.737573
MonthlyCharges                               2795.912582
Contract_Month-to-month                       409.776537
Contract_Two year                             386.138647
PaymentMethod_Electronic check                328.970764
OnlineSecurity_0                              326.284059
TechSupport_0                                 313.220029
InternetService_Fiber optic                   286.338882
OnlineBackup_0                                222.579066
StreamingTV_No internet service               222.180842
TechSupport_No internet service               222.180842
DeviceProtection_No internet service          222.180842
OnlineBackup_No internet service              222.180842
OnlineSecurity_No internet service            222.180842
StreamingMovies_No internet service           222.180842
InternetService_0                             222.180842
DeviceProtection_0             

In [17]:
for k in [10,15,20,'all']:
    model=make_pipeline(SelectKBest(chi2,k=k),LogisticRegression(max_iter=1000))
    scores_cv=cross_val_score(model,x_train,y_train,cv=5,scoring='f1')
    print(k,scores_cv.mean())

10 0.5815608536376702
15 0.5816884748313966
20 0.5801957393750496
all 0.5893266339743238


In [18]:
x=df[['TotalCharges','tenure','MonthlyCharges','Contract_Month-to-month','Contract_Two year','PaymentMethod_Electronic check',
'OnlineSecurity_1','TechSupport_1','InternetService_Fiber optic','StreamingTV_No internet service' ]].values
y=df['Churn'].values
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,shuffle=True)

In [19]:
lr=LogisticRegression()
lr.fit(x_train,y_train)
lr_pred=lr.predict(x_test)

In [20]:
x=df.drop('Churn',axis=1).values
y=df['Churn'].values
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,shuffle=True)

In [21]:
for k in [10,15,20,'all']:
    model=make_pipeline(SelectKBest(chi2,k=k),RandomForestClassifier(n_estimators=200,max_depth=None,min_samples_leaf=2,random_state=42,n_jobs=-1))
    scores_cv=cross_val_score(model,x_train,y_train,cv=5,scoring='f1')
    print(k,scores_cv.mean())

10 0.5597728223135748
15 0.5564153348588939
20 0.5550896756024049
all 0.561792474269736


In [22]:
rf=RandomForestClassifier(n_estimators=200,max_depth=None,min_samples_leaf=2,random_state=42,n_jobs=-1)
rf.fit(x_train,y_train)
rf_pred=rf.predict(x_test)

In [23]:
xgb=XGBClassifier(n_estimators=200,learning_rate=0.05,max_depth=4,subsample=0.8,gamma=0,colsample_bytree=0.8,eval_metric='logloss',
                        random_state=42,n_jobs=-1)
xgb.fit(x_train,y_train)
xgb_pred=xgb.predict(x_test)

In [25]:
lr_accuracy=accuracy_score(y_test,lr_pred)
lr_f1=f1_score(y_test,lr_pred)
lr_precision=precision_score(y_test,lr_pred)
lr_recall=recall_score(y_test,lr_pred)
print(f'logistic_accuracy:{lr_accuracy}\nlogistic_f1:{lr_f1}\nlogitic_precision:{lr_precision}\nlogistic_recall:{lr_recall}')
print(confusion_matrix(y_test,lr_pred))
print(classification_report(y_test,lr_pred))

logistic_accuracy:0.8119233498935415
logistic_f1:0.6120058565153733
logitic_precision:0.6741935483870968
logistic_recall:0.5603217158176944
[[935 101]
 [164 209]]
              precision    recall  f1-score   support

           0       0.85      0.90      0.88      1036
           1       0.67      0.56      0.61       373

    accuracy                           0.81      1409
   macro avg       0.76      0.73      0.74      1409
weighted avg       0.80      0.81      0.81      1409



In [26]:
rf_accuracy=accuracy_score(y_test,rf_pred)
rf_f1=f1_score(y_test,rf_pred)
rf_precision=precision_score(y_test,rf_pred)
rf_recall=recall_score(y_test,rf_pred)
print(f'random forest_accuracy:{rf_accuracy}\nrandom forest_f1:{rf_f1}\nrandom forest_precision:{rf_precision}\nrandom forest_recall:{rf_recall}')
print(confusion_matrix(y_test,rf_pred))
print(classification_report(y_test,rf_pred))

random forest_accuracy:0.7998580553584103
random forest_f1:0.5674846625766872
random forest_precision:0.6630824372759857
random forest_recall:0.4959785522788204
[[942  94]
 [188 185]]
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1036
           1       0.66      0.50      0.57       373

    accuracy                           0.80      1409
   macro avg       0.75      0.70      0.72      1409
weighted avg       0.79      0.80      0.79      1409



In [27]:
xgb_accuracy=accuracy_score(y_test,xgb_pred)
xgb_f1=f1_score(y_test,xgb_pred)
xgb_precision=precision_score(y_test,xgb_pred)
xgb_recall=recall_score(y_test,xgb_pred)
print(f'xgb_accuracy:{xgb_accuracy}\nxgb_f1:{xgb_f1}\nxgb_precision:{xgb_precision}\nxgb_recall:{xgb_recall}')
print(confusion_matrix(y_test,xgb_pred))
print(classification_report(y_test,xgb_pred))

xgb_accuracy:0.8112136266855926
xgb_f1:0.6017964071856288
xgb_precision:0.6813559322033899
xgb_recall:0.5388739946380697
[[942  94]
 [172 201]]
              precision    recall  f1-score   support

           0       0.85      0.91      0.88      1036
           1       0.68      0.54      0.60       373

    accuracy                           0.81      1409
   macro avg       0.76      0.72      0.74      1409
weighted avg       0.80      0.81      0.80      1409

